# **Install and Import Libraries**

> ##### **Make sure the secrets.env file is in the config folder. An example for secrets.env can be found in config/secrets_example.env file**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dotenv import load_dotenv
import os
import requests
from bs4 import BeautifulSoup
import time
import json
import re
from datetime import datetime
from neo4j import GraphDatabase

# load config
load_dotenv("../config/configOld.env")

# load secrets
load_dotenv("../config/secrets.env")

# **1. Fetch Course Links**

In [ ]:
BASE_URL = "https://mi.malax.fi"
COURSES_URL = f"{BASE_URL}/kurser/"
REQUEST_TIMEOUT = 10

def fetch_course_links():
    try:
        course_links = []
        page_num = 1
        max_pages = 10  # Limit to 10 pages to avoid infinite loops
        
        while page_num <= max_pages:
            # Build URL with page parameter
            if page_num == 1:
                current_url = COURSES_URL
            else:
                current_url = f"{COURSES_URL}?page={page_num}"
            
            print(f"Fetching page {page_num}: {current_url}")
            response = requests.get(current_url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Fetch courses on current page
            # Courses are now displayed as <article> elements within a generic container
            page_courses = 0
            articles = soup.find_all('article')
            print(f"Found {len(articles)} article elements on this page")
            
            for article in articles:
                # Find the course link within the article
                # Look for links that navigate to course detail pages
                course_link = article.find('a', class_='course-link')
                if not course_link:
                    # Fallback: look for any link that goes to /kurser/
                    course_link = article.find('a', href=lambda h: h and '/kurser/' in h)
                
                if course_link:
                    href = course_link.get('href')
                    if href:
                        full_link = BASE_URL + href if href.startswith('/') else href
                        # Avoid adding duplicates
                        if not any(c['link'] == full_link for c in course_links):
                            # Get title from heading within article (usually h2)
                            title_elem = article.find('h2')
                            title = title_elem.get_text(strip=True) if title_elem else course_link.get_text(strip=True)
                            
                            course_links.append({
                                'title': title,
                                'link': full_link
                            })
                            page_courses += 1
            
            print(f"Scraped {page_courses} courses from page {page_num}")
            
            # Stop if no new courses were scraped on this page
            if page_courses == 0:
                print("No new courses found on this page. Stopping pagination.")
                break
            
            # Try next page
            page_num += 1
            time.sleep(1)  # politely wait before next page fetch

        print(f"Total courses found: {len(course_links)}")
        return course_links
    except Exception as e:
        print(f"Error fetching courses: {e}")
        import traceback
        traceback.print_exc()
        return []

course_links = fetch_course_links()
print(f"Found {len(course_links)} courses.")
if course_links:
    print("\nFirst 3 courses:")
    for c in course_links[:3]:
        print(f"- {c['title']}: {c['link']}")


# **2. Scrape Complete Details from Course Pages**

In [ ]:
def scrape_course_details(url):
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        details = {}
        
        # Helper 1: For fields with <strong> tags (Kurskod, Lärare, Startdatum, etc.)
        def extract_field_by_label(label_text):
            for strong in soup.find_all('strong'):
                if label_text in strong.get_text(strip=True):
                    parent = strong.find_parent()
                    if parent:
                        all_text = parent.get_text(strip=True)
                        label_with_text = strong.get_text(strip=True)
                        if label_with_text in all_text:
                            remaining = all_text[all_text.find(label_with_text) + len(label_with_text):].strip()
                            return remaining
            return ""
            
        # Helper 2: For section headers that are just text (Plats, Anmälningstid, Information)
        def extract_next_sibling_text(label_text):
            node = soup.find(string=lambda s: s and s.strip() == label_text)
            if node:
                container = node.parent
                # Traverse up to find the element that has a meaningful visual sibling
                while container and container.name != 'body':
                    sibling = container.find_next_sibling()
                    while sibling:
                        text = sibling.get_text(strip=True)
                        if text:
                            return sibling, sibling.get_text(separator=' ', strip=True)
                        sibling = sibling.find_next_sibling()
                    container = container.parent
            return None, ""
        
        # Extract course code (Kurskod)
        details['coursecode'] = extract_field_by_label("Kurskod")
        
        # Extract price
        details['price'] = ""
        price_pattern = r"(\d+,\d{2}\s*€)"
        price_matches = re.findall(price_pattern, soup.get_text())
        if price_matches:
            details['price'] = price_matches[0].strip()
        
        # Extract teacher (Lärare)
        details['teacher'] = extract_field_by_label("Lärare")
        
        # Extract location (Plats)
        sibling, _ = extract_next_sibling_text("Plats")
        if sibling:
            parts = [text.strip() for text in sibling.stripped_strings if text.strip()]
            details['location'] = " ".join(parts)
        else:
            details['location'] = ""
        
        # Extract start date (Startdatum)
        details['start_date'] = None
        details['end_date'] = None
        start_date_str = extract_field_by_label("Startdatum")
        if start_date_str:
            try:
                start_date_obj = datetime.strptime(start_date_str, "%d.%m.%Y")
                details['start_date'] = start_date_obj.strftime("%Y-%m-%d")
            except ValueError:
                pass
        
        # Extract time information (Tid) and Day (Veckodag)
        times = extract_field_by_label("Tid")
        details['times'] = times
        day_of_week = extract_field_by_label("Veckodag")
        if day_of_week and times:
            details['times'] = f"{day_of_week} {times}".strip()
        
        # Extract description
        description = ""
        for p in soup.find_all('p'):
            p_text = p.get_text(strip=True)
            if len(p_text) > 80 and not any(label in p_text for label in ["Anmälningstid", "Tillfällen", "Information", "Plats"]):
                description = p_text
                break
        details['description'] = description
        
        # Extract signup times (Anmälningstid)
        details['signup_start'] = None
        details['signup_end'] = None
        details['signup_times'] = ""
        
        sibling, section_text = extract_next_sibling_text("Anmälningstid")
        if section_text:
            date_pattern = r"(\d{2}\.\d{2}\.\d{4})"
            time_pattern = r"(\d{2}:\d{2})"
            
            date_matches = re.findall(date_pattern, section_text)
            time_matches = re.findall(time_pattern, section_text)
            
            if len(date_matches) >= 1:
                try:
                    t_str = time_matches[0] if len(time_matches) >= 1 else "08:00"
                    dt_str = f"{date_matches[0]} {t_str}"
                    start_dt = datetime.strptime(dt_str, "%d.%m.%Y %H:%M")
                    details['signup_start'] = start_dt.strftime("%Y-%m-%dT%H:%M:%S")
                    details['signup_times'] = date_matches[0]
                except ValueError:
                    pass
            
            if len(date_matches) >= 2:
                try:
                    t_str = time_matches[1] if len(time_matches) >= 2 else "23:59"
                    dt_str = f"{date_matches[1]} {t_str}"
                    end_dt = datetime.strptime(dt_str, "%d.%m.%Y %H:%M")
                    details['signup_end'] = end_dt.strftime("%Y-%m-%dT%H:%M:%S")
                except ValueError:
                    pass
            
            if "Non-stop" in section_text or "nonstop" in section_text.lower():
                details['signup_times'] = "Non-stop"
                if details.get('start_date'):
                    details['signup_end'] = f"{details['start_date']}T23:59:59"
        
        # Extract language (Språk)
        details['languages'] = extract_field_by_label("Språk")
        details['spots_available'] = ""

        return details
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        import traceback
        traceback.print_exc()
        return None

# Use a slice to limit testing, e.g. course_links[:5] for faster run
# We map through our fetched link list
courses_data = []
for idx, course in enumerate(course_links): # process all courses
    print(f"[{idx+1}/{len(course_links)}] Scraping: {course['title']} ...", end="")
    details = scrape_course_details(course['link'])
    if details:
        full_course = {**course, **details}
        courses_data.append(full_course)
        print(" ✓")
    else:
        print(" ✗")
    time.sleep(1) # Be polite

if courses_data:
    print("\nSample Course Data:")
    print(json.dumps(courses_data[0], indent=2, ensure_ascii=False))

# **3. Connect to Neo4j Database**

In [ ]:
# Initialize Neo4j driver
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Test connection
try:
    with driver.session() as session:
        result = session.run("RETURN 1")
        print("✓ Successfully connected to Neo4j database")
except Exception as e:
    print(f"✗ Failed to connect to Neo4j: {e}")

# **4. Create Course Nodes in Neo4j**

In [ ]:
def create_or_update_course_node(driver, course):
    """
    Create or update a Course node in Neo4j.
    Uses the coursecode as a unique identifier.
    """
    with driver.session() as session:
        query = """
        MERGE (c:Course {coursecode: $coursecode})
        SET 
            c.title = $title,
            c.link = $link,
            c.price = $price,
            c.teacher = $teacher,
            c.location = $location,
            c.start_date = CASE WHEN $start_date IS NOT NULL THEN date($start_date) ELSE null END,
            c.end_date = CASE WHEN $end_date IS NOT NULL THEN date($end_date) ELSE null END,
            c.times = $times,
            c.signup_start = CASE WHEN $signup_start IS NOT NULL THEN localdatetime($signup_start) ELSE null END,
            c.signup_end = CASE WHEN $signup_end IS NOT NULL THEN localdatetime($signup_end) ELSE null END,
            c.spots_available = $spots_available,
            c.languages = $languages,
            c.description = $description,
            c.updated_at = datetime()
        RETURN c
        """
        
        result = session.run(
            query,
            coursecode=course.get("coursecode", ""),
            title=course.get("title", ""),
            link=course.get("link", ""),
            price=course.get("price", ""),
            teacher=course.get("teacher", ""),
            location=course.get("location", ""),
            start_date=course.get("start_date", None),
            end_date=course.get("end_date", None),
            times=course.get("times", ""),
            signup_start=course.get("signup_start", None),
            signup_end=course.get("signup_end", None),
            signup_times=course.get("signup_times", ""),
            spots_available=course.get("spots_available", ""),
            languages=course.get("languages", ""),
            description=course.get("description", "")
        )
        
        return result.single()

created_count = 0
print("Creating Course nodes in Neo4j...")
for course in courses_data:
    # Skip courses that might have failed to scrape proper details
    if not course.get("coursecode"):
        continue
    try:
        result = create_or_update_course_node(driver, course)
        if result:
            created_count += 1
            print(f"✓ Created/Updated: {course['title'][:50]}")
    except Exception as e:
        print(f"✗ Error creating course node: {e}")
        print(f"  Item: {course['title']}")

print(f"\nTotal items processed: {len(courses_data)}")
print(f"Successfully created/updated: {created_count}")

# **5. Verify and Close Database**

In [ ]:
def get_all_course_nodes(driver):
    query = """
    MATCH (c:Course)
    RETURN c.coursecode as code, c.title as title, c.teacher as teacher, c.location as location, c.start_date as start_date, c.end_date as end_date, c.signup_start as signup_start, c.signup_end as signup_end
    ORDER BY c.title ASC
    """
    with driver.session() as session:
        results = session.run(query)
        return [dict(record) for record in results]

all_courses = get_all_course_nodes(driver)
print(f"\n📚 All Course Nodes in Database ({len(all_courses)} total):\n")
for i, c in enumerate(all_courses[:10], 1): # list first 10
    print(f"{i}. {c['title']} (Code: {c['code']})")
    print(f"   Dates: {c['start_date']} to {c['end_date']}")
    print(f"   Signup: {c['signup_start']} to {c['signup_end']}")
    print(f"   Teacher: {c['teacher']}")
    print(f"   Location: {c['location']}")

driver.close()
print("\n✅ Database connection closed successfully!")